# Optional extension: switch an Agents SDK agent to Ollama

The agent and `Runner` are the same OpenAI Agents SDK primitives used in notebook 02. Only the **model connection** changes: this notebook sends Chat Completions requests to a local Ollama model. The example task is intentionally tiny so you can focus on the provider switch and its trace.

Default tracing goes to a **local MLflow store**, not the OpenAI Platform. No OpenAI API key is needed. An explicit opt-in below shows how to export the same Ollama-backed agent trace to OpenAI instead. Model calls still run in Ollama; they do not appear as OpenAI API model calls.

## Before you run

1. Install [Ollama](https://docs.ollama.com/) and ensure its local service is running (`ollama serve` if it is not already started).
2. Download a small chat model: `ollama pull qwen3:4b`.
3. In the course Python 3.12 `.venv`, install `requirements.txt` (which includes `openai-agents` and `mlflow`).
4. Run this notebook. It checks the Ollama service and model before calling the agent.

To try a different downloaded model, set `ANLP_OLLAMA_MODEL` before starting Jupyter, or set `ANLP_OLLAMA_SECOND_MODEL` to run the same task with two models. For example, `ollama pull llama3.2` and then set `ANLP_OLLAMA_SECOND_MODEL=llama3.2`. Choose a model suited to your machine.

The local API uses `http://127.0.0.1:11434/v1`. Ollama requires an API-key field in the OpenAI-compatible client, but ignores its value locally. [Ollama compatibility documentation](https://docs.ollama.com/api/openai-compatibility)

In [ ]:
import os
from pathlib import Path

import mlflow
import requests
from agents import (
    Agent, OpenAIChatCompletionsModel, Runner,
    set_trace_processors, set_tracing_export_api_key, trace,
)
from openai import AsyncOpenAI

OLLAMA_URL = os.getenv('ANLP_OLLAMA_URL', 'http://127.0.0.1:11434').rstrip('/')
PRIMARY_MODEL = os.getenv('ANLP_OLLAMA_MODEL', 'qwen3:4b')
SECOND_MODEL = os.getenv('ANLP_OLLAMA_SECOND_MODEL', '').strip()
TRACE_BACKEND = os.getenv('ANLP_TRACE_BACKEND', 'mlflow').lower()
EXPERIMENT_NAME = 'ANLP Session 9 - Ollama'

try:
    response = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
    response.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError('Ollama is unavailable. Start it with `ollama serve`.') from exc

available_models = {entry['name'] for entry in response.json()['models']}
for name in (PRIMARY_MODEL, SECOND_MODEL):
    if name and name not in available_models and f'{name}:latest' not in available_models:
        raise RuntimeError(f'Model {name!r} is not installed. Run `ollama pull {name}`.')
print(f'Ollama is ready. Primary model: {PRIMARY_MODEL}')

## Choose where traces go

The default `mlflow` mode stores traces on this computer. `mlflow.openai.autolog()` instruments the Agents SDK and Ollama's OpenAI-compatible calls; replacing the Agents SDK's default trace processor prevents an accidental upload to OpenAI. MLflow may capture prompts and outputs, so keep this store private.

After running the agent, run the `mlflow ui --backend-store-uri` command printed below in a separate terminal, then open `http://127.0.0.1:5000`. If port 5000 is occupied, add `--port 5001` and open `http://127.0.0.1:5001`.

To opt in to the [OpenAI Traces dashboard](https://platform.openai.com/traces) instead, set `ANLP_TRACE_BACKEND=openai` **and** set `OPENAI_API_KEY` before starting Jupyter. The key is used only for trace export, not for Ollama model inference. Do not put a key in the notebook. [OpenAI tracing documentation](https://openai.github.io/openai-agents-python/tracing/) · [MLflow Agents SDK tracing documentation](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/openai-agent/)

In [ ]:
if TRACE_BACKEND == 'mlflow':
    # Use a local SQLite database outside the repository; no cloud account is needed.
    local_db = Path(os.getenv('ANLP_MLFLOW_DB', Path.home() / '.anlp-session9' / 'mlflow.db'))
    local_db.parent.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', f'sqlite:///{local_db.as_posix()}'))
    mlflow.set_experiment(EXPERIMENT_NAME)
    set_trace_processors([])  # Do not upload the SDK trace to OpenAI.
    mlflow.openai.autolog()
    print(f'Local MLflow tracking URI: {mlflow.get_tracking_uri()}')
    print(f'View traces: mlflow ui --backend-store-uri "{mlflow.get_tracking_uri()}"')
elif TRACE_BACKEND == 'openai':
    tracing_key = os.getenv('OPENAI_API_KEY')
    if not tracing_key:
        raise RuntimeError('Set OPENAI_API_KEY to export traces to OpenAI.')
    set_tracing_export_api_key(tracing_key)
    print('Agent traces will appear at https://platform.openai.com/traces')
else:
    raise ValueError('ANLP_TRACE_BACKEND must be mlflow or openai')



## Same agent, different model

`OpenAIChatCompletionsModel` adapts an OpenAI-compatible Ollama endpoint to the Agents SDK. The placeholder key below is **not** an OpenAI credential. If you run a second model, only `model_name` changes; the instructions, `Runner`, task, and tracing setup stay the same.

In [ ]:
async def run_with_model(model_name: str) -> str:
    async with AsyncOpenAI(base_url=f'{OLLAMA_URL}/v1', api_key='ollama') as client:
        agent = Agent(
            name='One-sentence explainer',
            instructions='Answer in one clear sentence.',
            model=OpenAIChatCompletionsModel(model=model_name, openai_client=client),
        )
        with trace(workflow_name=f'Session 9 local model: {model_name}'):
            result = await Runner.run(agent, 'What is an AI agent?')
        return str(result.final_output)

for model_name in (PRIMARY_MODEL, SECOND_MODEL):
    if model_name:
        answer = await run_with_model(model_name)
        print(f'Model: {model_name}\nAnswer: {answer}\n')

## What changed?

The model endpoint and model name changed; the `Agent`, `Runner`, task, and trace destination did not. With local MLflow tracing, model calls and trace data stay on this computer. If you explicitly choose OpenAI tracing, trace content is sent to OpenAI even though inference still runs in Ollama. Other providers may differ in tool calling, structured outputs, and streaming support, so test those features separately before swapping them into a more complex notebook.